# NiyamTrace-X Wave 3 / Experiment 10 — BFCL V4 + MLCL External Multi-Model Validation

This notebook runs a **real external function-calling benchmark** across multiple model endpoints using EvalScope's BFCL-v4 adapter, then converts the official outputs into a NiyamTrace-X-oriented effect-boundary analysis.

### Design principles
- The official BFCL score remains the primary external metric; we do not replace its grader.
- A separate **effect-ceiling proxy** classifies predicted tool-call sets as exact/safe, missing/clarify, or expanded/block relative to any reference calls exposed by the official review artifacts.
- Mapping coverage is reported. Missing reference fields are `UNSUPPORTED_FOR_EFFECT_MAPPING`, never guessed.
- QUICK / STANDARD / FULL modes are provided.
- Arbitrary OpenAI-compatible endpoints are supported, so Qwen, GPT-OSS, DeepSeek, Llama, GLM or hosted APIs can share one harness.
- MLCL is used only if an official/local release is supplied. No synthetic reconstruction is substituted for the ACL benchmark.


In [ ]:
import importlib.util,subprocess,sys
pkgs=['evalscope','pandas','numpy','matplotlib']
# BFCL-v4 adapter currently documents this bfcl-eval release; pin it for reproducibility.
subprocess.check_call([sys.executable,'-m','pip','install','-q','evalscope','bfcl-eval==2025.10.27.1','pandas','numpy','matplotlib'])


In [ ]:
from pathlib import Path
import os, sys, json, re, math, time, random, hashlib, zipfile, shutil, subprocess, statistics, tempfile, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED=20260911
random.seed(SEED); np.random.seed(SEED)
BASE=Path('/content') if Path('/content').exists() else Path('/mnt/data')
RESULTS=BASE/'niyamtrace_q1_wave3_results'
RESULTS.mkdir(parents=True,exist_ok=True)
print('BASE:',BASE)
print('RESULTS:',RESULTS)


In [ ]:
# Multi-model configuration. For a formal run, use at least 3 independent model families.
# Preferred: set NTX_MODELS_JSON in the environment so credentials never enter the notebook.
# Format:
# [
#   {"label":"qwen35-122b","model":"Qwen/Qwen3.5-122B-A10B-FP8","api_url":"http://HOST:PORT/v1","api_key":"EMPTY"},
#   {"label":"gpt-oss-120b","model":"openai/gpt-oss-120b","api_url":"http://HOST:PORT/v1","api_key":"EMPTY"}
# ]
raw=os.getenv('NTX_MODELS_JSON','').strip()
MODELS=json.loads(raw) if raw else []
for m in MODELS:
    for k in ['label','model','api_url']:
        if not m.get(k): raise ValueError(f'Model spec missing {k}: {m}')
    m.setdefault('api_key','EMPTY')
print('Configured model endpoints:', [m['label'] for m in MODELS])
if not MODELS:
    print('No model endpoints configured. Benchmark discovery/mapping cells can still run; model inference cells will record SKIPPED_NO_MODEL_CONFIG rather than invent results.')


In [ ]:
MODE=os.getenv('NTX_RUN_MODE','QUICK').upper()
assert MODE in {'QUICK','STANDARD','FULL'}
LIMIT={'QUICK':10,'STANDARD':100,'FULL':None}[MODE]
SUBSETS={'QUICK':['simple_python','simple_java'],'STANDARD':['simple_python','simple_java','simple_javascript','parallel','multiple','irrelevance'],'FULL':None}[MODE]
print('Run mode:',MODE,'limit:',LIMIT,'subsets:',SUBSETS or 'ALL OFFICIAL BFCL V4 SUBSETS')


In [ ]:
# Run BFCL-v4 with the official EvalScope adapter. Each model gets its own output tree.
from evalscope import TaskConfig, run_task
run_records=[]
for m in MODELS:
    out=RESULTS/f"exp10_bfcl_{m['label']}"
    out.mkdir(parents=True,exist_ok=True)
    dargs={'bfcl_v4':{'is_fc_model':True}}
    if SUBSETS: dargs['bfcl_v4']['subset_list']=SUBSETS
    if os.getenv('SERPAPI_API_KEY'): dargs['bfcl_v4']['SERPAPI_API_KEY']=os.getenv('SERPAPI_API_KEY')
    cfg=TaskConfig(model=m['model'],model_id=m['label'],api_url=m['api_url'],api_key=m.get('api_key','EMPTY'),eval_type='openai_api',datasets=['bfcl_v4'],dataset_args=dargs,generation_config={'temperature':0},eval_batch_size=1,limit=LIMIT,work_dir=str(out),seed=SEED,ignore_errors=False)
    t=time.time(); status='OK'; err=''
    try: run_task(task_cfg=cfg)
    except Exception as e: status='ERROR'; err=repr(e)
    run_records.append({'benchmark':'BFCL-v4','model':m['label'],'status':status,'seconds':time.time()-t,'error':err,'output_dir':str(out)})
if not MODELS:
    run_records.append({'benchmark':'BFCL-v4','model':'NONE','status':'SKIPPED_NO_MODEL_CONFIG','seconds':0,'error':'Set NTX_MODELS_JSON','output_dir':''})
run_df=pd.DataFrame(run_records); run_df.to_csv(RESULTS/'exp10_run_status.csv',index=False); display(run_df)


In [ ]:
# Parse all official reports into one model/subset table.
def walk_json(root):
    for p in Path(root).rglob('*.json'):
        try: yield p,json.loads(p.read_text())
        except Exception: pass
def scalar_rows(obj,prefix=''):
    rows=[]
    if isinstance(obj,dict):
        for k,v in obj.items(): rows.extend(scalar_rows(v,f'{prefix}.{k}' if prefix else str(k)))
    elif isinstance(obj,(int,float,str,bool)) or obj is None: rows.append((prefix,obj))
    return rows
report_rows=[]
for m in MODELS:
    root=RESULTS/f"exp10_bfcl_{m['label']}"
    for p,obj in walk_json(root):
        if 'report' not in str(p).lower(): continue
        for key,val in scalar_rows(obj):
            if isinstance(val,(int,float)) and any(x in key.lower() for x in ['score','accuracy','acc']):
                report_rows.append({'model':m['label'],'file':str(p.relative_to(root)),'metric_path':key,'value':val})
reports=pd.DataFrame(report_rows)
reports.to_csv(RESULTS/'exp10_bfcl_report_metrics.csv',index=False)
display(reports.head(50))


In [ ]:
# Standardize review/prediction artifacts and compute an effect-ceiling proxy only when both predicted and reference tool calls are discoverable.
def rec_find(o,keys):
    if isinstance(o,dict):
        for k,v in o.items():
            if k.lower() in keys: return v
        for v in o.values():
            x=rec_find(v,keys)
            if x is not None:return x
    if isinstance(o,list):
        for v in o:
            x=rec_find(v,keys)
            if x is not None:return x
    return None
def tool_calls(o):
    out=[]
    def visit(x):
        if isinstance(x,dict):
            if 'function' in x and isinstance(x['function'],dict) and x['function'].get('name'):
                f=x['function']; out.append((str(f.get('name')),json.dumps(f.get('arguments',{}),sort_keys=True,default=str)))
            elif x.get('name') and any(k in x for k in ['arguments','args','parameters']):
                out.append((str(x['name']),json.dumps(x.get('arguments',x.get('args',x.get('parameters',{}))),sort_keys=True,default=str)))
            for v in x.values(): visit(v)
        elif isinstance(x,list):
            for v in x: visit(v)
    visit(o); return sorted(set(out))
std=[]
for m in MODELS:
    root=RESULTS/f"exp10_bfcl_{m['label']}"
    files=list(root.rglob('reviews/**/*.jsonl'))+list(root.rglob('reviews/*.jsonl'))
    if not files: files=list(root.rglob('*.jsonl'))
    seen=set()
    for p in files:
        for i,line in enumerate(p.read_text(errors='ignore').splitlines()):
            try:o=json.loads(line)
            except: continue
            case=str(rec_find(o,{'id','case_id','question_id','sample_id'}) or f'{p.name}:{i}')
            pred=rec_find(o,{'prediction','pred','model_output','response','output'})
            ref=rec_find(o,{'reference','ground_truth','answer','target','expected'})
            pc,rc=tool_calls(pred),tool_calls(ref)
            key=(m['label'],case,str(p));
            if key in seen:continue
            seen.add(key)
            supported=bool(rc)
            ps,rs=set(pc),set(rc)
            if supported:
                if ps==rs: verdict='ALLOW'; expansion=False; missing=False
                elif ps-rs: verdict='BLOCK'; expansion=True; missing=bool(rs-ps)
                else: verdict='CLARIFY'; expansion=False; missing=True
            else: verdict='UNSUPPORTED'; expansion=False; missing=False
            score=rec_find(o,{'score','correct','passed','is_correct'})
            std.append({'benchmark':'BFCL-v4','model':m['label'],'case_id':case,'source_file':str(p.relative_to(root)),'mapping_supported':supported,'pred_call_count':len(ps),'ref_call_count':len(rs),'effect_proxy_verdict':verdict,'effect_expansion_proxy':expansion,'missing_effect_proxy':missing,'official_item_score':score})
std=pd.DataFrame(std); std.to_csv(RESULTS/'exp10_standardized_cases.csv',index=False)
if len(std):
    cov=std.groupby('model').agg(n=('case_id','size'),mapping_coverage=('mapping_supported','mean'),effect_expansion_rate=('effect_expansion_proxy','mean')).reset_index(); display(cov); cov.to_csv(RESULTS/'exp10_effect_mapping_summary.csv',index=False)
else:
    pd.DataFrame(columns=['model','n','mapping_coverage','effect_expansion_rate']).to_csv(RESULTS/'exp10_effect_mapping_summary.csv',index=False)


In [ ]:
# MLCL availability gate. We will only use a genuine supplied release; otherwise we record the gap.
mlcl_path=os.getenv('MLCL_LOCAL_PATH','').strip()
status={'benchmark':'MLCL','status':'NOT_RUN_NO_OFFICIAL_RELEASE_CONFIGURED','path':mlcl_path}
if mlcl_path and Path(mlcl_path).exists():
    p=Path(mlcl_path); status['status']='LOCAL_RELEASE_FOUND'; status['files']=sum(1 for _ in p.rglob('*') if _.is_file()) if p.is_dir() else 1
(RESULTS/'exp10_mlcl_availability.json').write_text(json.dumps(status,indent=2)); print(status)


In [ ]:
# Publication summary and manifest.
if len(reports):
    top=reports.groupby('model').value.max().sort_values(ascending=False)
    fig,ax=plt.subplots(figsize=(7,4)); top.plot(kind='bar',ax=ax); ax.set_ylabel('Maximum reported BFCL metric'); ax.set_title('BFCL-v4 multi-model external evaluation'); fig.tight_layout(); fig.savefig(RESULTS/'exp10_bfcl_multimodel_summary.png',dpi=220); plt.show()
manifest={'experiment':'NTX-Q1-10','mode':MODE,'models':[{k:v for k,v in m.items() if k!='api_key'} for m in MODELS],'bfcl_adapter':'EvalScope bfcl_v4 + bfcl-eval 2025.10.27.1','mlcl_status':status,'result_files':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in RESULTS.glob('exp10_*') if p.is_file()}}
(RESULTS/'exp10_manifest.json').write_text(json.dumps(manifest,indent=2))


In [ ]:
# FINAL CELL — package every result from this experiment and download it.
PREFIX='exp10_'
ZIP_OUT=BASE/'NTX_Q1_10_BFCL_MLCL_MULTIMODEL_RESULTS.zip'
with zipfile.ZipFile(ZIP_OUT,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RESULTS.rglob('*')):
        if p.is_file() and p.name.startswith(PREFIX):
            z.write(p,arcname=str(p.relative_to(RESULTS)))
sha=hashlib.sha256(ZIP_OUT.read_bytes()).hexdigest()
print('Created:',ZIP_OUT)
print('SHA-256:',sha)
print('Size MiB:',round(ZIP_OUT.stat().st_size/1024**2,3))
try:
    from google.colab import files
    files.download(str(ZIP_OUT))
except Exception:
    print('Not running in Colab. ZIP is available at',ZIP_OUT)
